In [1]:
import sys, os, warnings, subprocess
subprocess.run(["pip", "install", ".", "--quiet"], check=True, cwd="/mnt/storage_ssd_raid/research_projects/intrinsic_neural_timescales")
warnings.simplefilter("ignore", UserWarning)

import numpy as np
import pandas as pd

from src.data import DataLoader

  DEPRECATION: Building 'src' using the legacy setup.py bdist_wheel mechanism, which will be removed in a future version. pip 25.3 will enforce this behaviour change. A possible replacement is to use the standardized build interface by setting the `--use-pep517` option, (possibly combined with `--no-build-isolation`), or adding a `pyproject.toml` file to the source tree of 'src'. Discussion can be found at https://github.com/pypa/pip/issues/6334


In [2]:
# ── Regenerate PNC subject_ids_lowmotion.txt with explicit FD threshold ────────
FD_THRESHOLD = 0.2

pnc_loader = DataLoader(dataset='PNC', verbose=False)
base = pnc_loader.base_dir

# Load original subject list (strip sub- prefix to get bare IDs)
all_subs = np.loadtxt(str(base / 'subject_ids.txt'), dtype=str).tolist()
all_subs_bare = [s.replace('sub-', '') for s in all_subs]
print(f"Original subject list:  {len(all_subs_bare)} subjects")

# Load QC file; keep only the two tasks of interest
qc = pd.read_csv(base / 'study-PNC_desc-functional_qc.tsv', sep='\t',
                 dtype={'participant_id': str})
qc = qc[qc['task'].isin(['rest', 'frac2back'])]

# Mean FD per subject per task (collapses multiple runs/acqs if present)
fd = qc.groupby(['participant_id', 'task'])['meanFD'].mean().unstack('task')

# Exclude if FD > threshold in either task; require both tasks present
passes = (fd <= FD_THRESHOLD).all(axis=1) & fd.notna().all(axis=1)
passing = fd.index[passes].astype(str).tolist()

# Restrict to subjects that were in the original cohort
passing = sorted(s for s in passing if s in set(all_subs_bare))

n_excluded = len(all_subs_bare) - len(passing)
print(f"FD <= {FD_THRESHOLD} mm, both tasks present:  {len(passing)} subjects retained")
print(f"Excluded:               {n_excluded} subjects")

# Write out preserving the original sub- prefix format
out_path = base / 'subject_ids_lowmotion.txt'
with open(out_path, 'w') as f:
    f.write('\n'.join(f'sub-{s}' for s in passing) + '\n')
print(f"Written: {out_path}")


Original subject list:  1601 subjects
FD <= 0.2 mm, both tasks present:  1095 subjects retained
Excluded:               506 subjects
Written: /mnt/storage_ssd_raid/research_data/RBC/PNC/subject_ids_lowmotion.txt


In [3]:
# ── Sample demographics summary for methods paragraph ─────────────────────────
# use_for_total=False: this subsample is subsumed by another row, so exclude from combined counts

REGISTRY_CONFIGS = [
    dict(label='HCP-YA',      dataset='HCP-YA', registry='scan_registry_HCP-YA.csv',       use_for_total=True),
    dict(label='HCP-YA (7T)', dataset='HCP-YA', registry='scan_registry_HCP-YA_multi.csv', use_for_total=False),
    dict(label='WAND',        dataset='WAND',    registry='scan_registry_WAND.csv',          use_for_total=True),
    dict(label='HCP-D',       dataset='HCP-D',   registry='scan_registry_HCP-D.csv',         use_for_total=True),
    dict(label='PNC',         dataset='PNC',     registry='scan_registry_PNC.csv',           use_for_total=True),
    dict(label='Penn_LEAD',   dataset='Penn_LEAD', registry='scan_registry_Penn_LEAD.csv',   use_for_total=True),
]

rows = []
loaders = {}
for cfg in REGISTRY_CONFIGS:
    dataset = cfg['dataset']
    if dataset not in loaders:
        loaders[dataset] = DataLoader(dataset=dataset, verbose=False)
    loader = loaders[dataset]

    # Included subject IDs from scan registry CSV
    reg = pd.read_csv(str(loader.output_dir / cfg['registry']), index_col=0, dtype=str)
    included = set(reg['sub'].unique())
    n = len(included)

    # Match to demographics
    if loader.df_behavior is not None:
        df_beh = loader.df_behavior.copy()
        df_beh.index = df_beh.index.astype(str)
        df_m = df_beh.loc[df_beh.index.intersection(included)]
    else:
        # No handler (e.g. WAND) — load participants.tsv from dataset base dir
        try:
            df_m = pd.read_csv(loader.base_dir / 'participants.tsv', sep='\t')
            df_m['subject_id'] = df_m['participant_id'].str.replace('sub-', '', regex=False)
            df_m = df_m.set_index('subject_id').loc[lambda d: d.index.isin(included)]
            sex_col = next((c for c in df_m.columns if c.lower() in ('sex', 'gender')), None)
            if sex_col:
                df_m = df_m.rename(columns={sex_col: 'sex'})
                df_m['sex'] = df_m['sex'].map(
                    lambda v: 0 if str(v).lower() in ('f', 'female') else
                              1 if str(v).lower() in ('m', 'male') else np.nan)
        except Exception as e:
            print(f'  {dataset}: participants.tsv not loaded ({e})')
            df_m = pd.DataFrame()

    # Age (all sources store age in years)
    if 'age' in df_m.columns:
        ages      = df_m['age'].dropna()
        age_min   = ages.min()
        age_max   = ages.max()
        age_range = f"{age_min:.1f}-{age_max:.1f}"
        age_ms    = f"{ages.mean():.1f} +/- {ages.std():.1f}"
    else:
        age_min = age_max = np.nan
        age_range = age_ms = 'N/A'

    # Sex: 0=Female, 1=Male across all sources
    if 'sex' in df_m.columns:
        n_female   = int((df_m['sex'] == 0).sum())
        pct_female = 100 * n_female / len(df_m)
    else:
        n_female = pct_female = np.nan

    # Mean FD: HCP-D has a single meanFD column; PNC has per-task columns
    if 'meanFD' in df_m.columns:
        fd = df_m['meanFD'].dropna()
    elif {'meanFD_rest', 'meanFD_frac2back'}.issubset(df_m.columns):
        fd = df_m[['meanFD_rest', 'meanFD_frac2back']].mean(axis=1).dropna()
    else:
        fd = pd.Series(dtype=float)

    if len(fd):
        mean_fd_str  = f"{fd.mean():.3f} +/- {fd.std():.3f}"
        range_fd_str = f"{fd.min():.3f}-{fd.max():.3f}"
    else:
        mean_fd_str = range_fd_str = 'N/A'

    rows.append(dict(
        label=cfg['label'], n=n, use_for_total=cfg['use_for_total'],
        age_min=age_min, age_max=age_max,
        age_range=age_range, age_ms=age_ms,
        n_female=n_female, pct_female=pct_female,
        mean_fd_str=mean_fd_str, range_fd_str=range_fd_str,
    ))

df_sum = pd.DataFrame(rows)
df_total = df_sum[df_sum['use_for_total']]

# Combined totals (larger HCP-YA subsample only; NaN-safe for female count)
n_total        = int(df_total['n'].sum())
age_min_all    = df_total['age_min'].min()
age_max_all    = df_total['age_max'].max()
n_female_all   = df_total['n_female'].sum()
pct_female_all = 100 * n_female_all / n_total

# ── Table ─────────────────────────────────────────────────────────────────────
hdr = f"{'Dataset':<14}  {'N':>5}  {'Age range':>12}  {'Age M+/-SD':>14}  {'N female':>8}  {'% female':>8}  {'Mean FD (M+/-SD)':>18}  {'FD range':>14}"
sep = '-' * len(hdr)
print(hdr)
print(sep)
for _, r in df_sum.iterrows():
    nf  = f"{int(r['n_female'])}" if not np.isnan(r['n_female']) else 'N/A'
    pf  = f"{r['pct_female']:.1f}" if not np.isnan(r['pct_female']) else 'N/A'
    tag = '' if r['use_for_total'] else '*'
    print(f"{r['label'] + tag:<14}  {r['n']:>5}  {r['age_range']:>12}  {r['age_ms']:>14}  {nf:>8}  {pf:>8}  {r['mean_fd_str']:>18}  {r['range_fd_str']:>14}")
print(sep)
print('* excluded from combined totals (subsumed by HCP-YA)')
print('  Mean FD for PNC is the per-subject average of rest and frac2back FD')
print()
print(f"N_TOTAL     = {n_total}")
print(f"AGE_MIN     = {age_min_all:.1f}")
print(f"AGE_MAX     = {age_max_all:.1f}")
print(f"N_FEMALE    = {int(n_female_all)}")
print(f"PCT_FEMALE  = {pct_female_all:.1f}")


Dataset             N     Age range      Age M+/-SD  N female  % female    Mean FD (M+/-SD)        FD range
-----------------------------------------------------------------------------------------------------------
HCP-YA            966     22.0-37.0    28.7 +/- 3.7       513      53.1                 N/A             N/A
HCP-YA (7T)*      163     22.0-36.0    29.5 +/- 3.3        98      60.1                 N/A             N/A
WAND              160     18.3-63.7   29.7 +/- 10.6        97      60.6                 N/A             N/A
HCP-D             552      8.1-21.9    14.9 +/- 3.9       299      54.2     0.128 +/- 0.033     0.033-0.200
PNC               998      8.2-23.0    16.0 +/- 3.3       549      55.0     0.065 +/- 0.032     0.016-0.176
Penn_LEAD          43      8.7-15.9    12.5 +/- 2.0        18      41.9                 N/A             N/A
-----------------------------------------------------------------------------------------------------------
* excluded from combined tot